Задание Light — версия на PyTorch

Выполните подробную работу с параметрами словаря и формированием гиперпараметров нейронной сети. Создайте 9 нейросетей с различными гиперпараметрами (см. пунтк 2 и 3)

 Для этого необходимо:

  1. Воссоздать ноутбук, аналогичный ноутбуку практической части №1, загрузив при этом необходимую нам базу (код уже доступен в ноутбуке).

  2. Задать в ноутбуке следующие параметры для размера словаря, ширины окна и шага:

    - Размер словаря - от 10000 до 20000 (выбрать меньшее значение диапазона, если будет перегрузка ОЗУ и перезапуск подключения к Colaboratory)
    - Ширина окна - от 1000 до 2000
    - Шаг - от 100 до 500 (на обучение лучше влияет наименьший шаг, но это может перегрузить ОЗУ).

  3. Создать архитектуру сети и задать гиперпараметры. Можно воспользоваться шаблоном:
  
   - Добавьте модель прямого распространения **Sequential()**
   - Добавьте один или несколько полносвязных (**Dense**) слоёв
   - Добавьте слои **Dropout()** и **BatchNormalization()**
   - Добавьте выходной полносвязный слой с количеством нейронов, соответствующим количеству классов (число писателей)
  
   Напомним, что точность сети можно проверить по значению показателя 'val_accuracy' на конце каждой эпохи.
   

**Изменение:** исходная версия на TensorFlow/Keras переписана на PyTorch. Подготовка данных сохранена по смыслу: тексты разбиваются на окна, каждое окно переводится в бинарный мешок слов, затем обучается полносвязная нейронная сеть.


In [ ]:
# импорт библиотек
import os
import re
import time
import zipfile
import random
import shutil
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

%matplotlib inline

# фиксация случайности, чтобы результаты меньше прыгали при повторном запуске
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2)

# настройки для более стабильного результата
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство для обучения:', DEVICE)


In [ ]:
# загрузка и распаковка базы
DATA_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip'
ARCHIVE_NAME = 'writers.zip'
FILE_DIR = 'writers'

# загрузка архива, если его еще нет в папке
if not os.path.exists(ARCHIVE_NAME):
    urllib.request.urlretrieve(DATA_URL, ARCHIVE_NAME)
    print('Архив загружен:', ARCHIVE_NAME)
else:
    print('Архив уже загружен:', ARCHIVE_NAME)

# очистка старой папки, чтобы при повторном запуске файлы не смешивались
if os.path.exists(FILE_DIR):
    shutil.rmtree(FILE_DIR)

# распаковка через zipfile вместо команды unzip
with zipfile.ZipFile(ARCHIVE_NAME, 'r') as archive:
    archive.extractall(FILE_DIR)

print('Файлы в папке:')
print(os.listdir(FILE_DIR)[:10])


In [ ]:
# загрузка текстов писателей
SIG_TRAIN = 'обучающая' # признак обучающей выборки в названии файла
SIG_TEST = 'тестовая'   # признак тестовой выборки в названии файла

CLASS_LIST = []
text_train = []
text_test = []

# чтение файла с запасным вариантом кодировки
def read_text(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except UnicodeDecodeError:
        with open(path, 'r', encoding='cp1251', errors='ignore') as f:
            return f.read()

# сортировка нужна, чтобы порядок классов был стабильным
for file_name in sorted(os.listdir(FILE_DIR)):
    # выделение автора и типа выборки из имени файла
    match = re.match(r'\((.+)\)\s+(\S+)_', file_name)

    if match is None:
        continue

    class_name = match.group(1)
    subset_name = match.group(2).lower()

    is_train = SIG_TRAIN in subset_name
    is_test = SIG_TEST in subset_name

    if not (is_train or is_test):
        continue

    if class_name not in CLASS_LIST:
        CLASS_LIST.append(class_name)
        text_train.append('')
        text_test.append('')
        print(f'добавление класса: {class_name}')

    class_index = CLASS_LIST.index(class_name)
    file_path = os.path.join(FILE_DIR, file_name)
    text = read_text(file_path).replace('\n', ' ')

    if is_train:
        text_train[class_index] += ' ' + text
    else:
        text_test[class_index] += ' ' + text

CLASS_COUNT = len(CLASS_LIST)

print('Количество классов:', CLASS_COUNT)
print('Классы:', CLASS_LIST)
print('Длина обучающих текстов:', [len(text) for text in text_train])
print('Длина проверочных текстов:', [len(text) for text in text_test])


In [ ]:
# класс подготовки данных без TensorFlow/Keras
class DataProcessor:
    def __init__(self, vocab_size, win_size, win_hop):
        self.vocab_size = vocab_size # размер словаря
        self.win_size = win_size     # ширина окна
        self.win_hop = win_hop       # шаг окна
        self.word_index = {}         # словарь слово -> индекс

    # простая токенизация: оставляем буквы и цифры
    def tokenize(self, text):
        return re.findall(r'[а-яёa-z0-9]+', text.lower())

    # построение словаря по обучающим текстам
    def fit_tokenizer(self, train_texts):
        if self.vocab_size < 3:
            raise ValueError('VOCAB_SIZE должен быть не меньше 3')

        counter = Counter()
        for text in train_texts:
            counter.update(self.tokenize(text))

        # 0 оставлен под пустое значение, 1 — unknown
        self.word_index = {'unknown': 1}

        # vocab_size - 2, потому что индексы 0 и 1 зарезервированы
        for index, (word, _) in enumerate(counter.most_common(self.vocab_size - 2), start=2):
            self.word_index[word] = index

    # перевод текстов в последовательности индексов
    def texts_to_sequences(self, texts):
        sequences = []

        for text in texts:
            tokens = self.tokenize(text)
            sequence = [self.word_index.get(token, 1) for token in tokens]
            sequences.append(sequence)

        return sequences

    # нарезка последовательности на окна
    def make_windows(self, sequence):
        windows = []

        for start in range(0, len(sequence) - self.win_size + 1, self.win_hop):
            windows.append(sequence[start:start + self.win_size])

        # защита от пустой выборки, если текст оказался короче окна
        if len(windows) == 0 and len(sequence) > 0:
            windows.append(sequence)

        return windows

    # создание примеров и правильных ответов
    def vectorize_sequence(self, sequences):
        x_data = []
        y_data = []

        for class_index, sequence in enumerate(sequences):
            windows = self.make_windows(sequence)
            x_data.extend(windows)
            y_data.extend([class_index] * len(windows))

        return x_data, np.array(y_data, dtype='int64')

    # перевод последовательностей в бинарный мешок слов
    def sequences_to_matrix(self, sequences):
        matrix = np.zeros((len(sequences), self.vocab_size), dtype='float32')

        for row_index, sequence in enumerate(sequences):
            for token_index in set(sequence):
                if 0 <= token_index < self.vocab_size:
                    matrix[row_index, token_index] = 1.0

        return matrix

    # полная подготовка обучающей и проверочной выборки
    def prepare(self, train_texts, test_texts):
        self.fit_tokenizer(train_texts)

        train_sequences = self.texts_to_sequences(train_texts)
        test_sequences = self.texts_to_sequences(test_texts)

        x_train, y_train = self.vectorize_sequence(train_sequences)
        x_test, y_test = self.vectorize_sequence(test_sequences)

        x_train_bow = self.sequences_to_matrix(x_train)
        x_test_bow = self.sequences_to_matrix(x_test)

        print('VOCAB_SIZE:', self.vocab_size)
        print('WIN_SIZE:', self.win_size)
        print('WIN_HOP:', self.win_hop)
        print('x_train_bow:', x_train_bow.shape)
        print('y_train:', y_train.shape)
        print('x_test_bow:', x_test_bow.shape)
        print('y_test:', y_test.shape)

        return x_train_bow, y_train, x_test_bow, y_test


# модель на PyTorch
class WriterClassifier(nn.Module):
    def __init__(self, input_dim, class_count):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, class_count)
        )

    def forward(self, x):
        return self.net(x)


# создание модели
def create_model(input_dim, class_count):
    model = WriterClassifier(input_dim=input_dim, class_count=class_count)
    return model.to(DEVICE)


# расчет ошибки и точности на одном проходе
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            if is_train:
                loss.backward()
                optimizer.step()

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total_count += batch_size

    avg_loss = total_loss / total_count
    accuracy = total_correct / total_count

    return avg_loss, accuracy


# обучение модели PyTorch
def train_model(model, x_train, y_train, x_test, y_test, epochs=12, batch_size=128):
    train_dataset = TensorDataset(
        torch.tensor(x_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )

    test_dataset = TensorDataset(
        torch.tensor(x_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.long)
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    history = {
        'accuracy': [],
        'val_accuracy': [],
        'loss': [],
        'val_loss': []
    }

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)

        with torch.no_grad():
            val_loss, val_acc = run_epoch(model, test_loader, criterion, optimizer=None)

        history['loss'].append(train_loss)
        history['accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        print(
            f'Эпоха {epoch:02d}/{epochs} | '
            f'loss: {train_loss:.4f} | accuracy: {train_acc:.4f} | '
            f'val_loss: {val_loss:.4f} | val_accuracy: {val_acc:.4f}'
        )

    return history


# график обучения одной модели
def plot_history(history, vocab_size, win_size, win_hop):
    epochs = range(1, len(history['accuracy']) + 1)

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['accuracy'], label='точность обучения')
    plt.plot(epochs, history['val_accuracy'], label='точность проверки')
    plt.title(f'точность: словарь={vocab_size}, окно={win_size}, шаг={win_hop}')
    plt.xlabel('эпоха')
    plt.ylabel('точность')
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['loss'], label='ошибка обучения')
    plt.plot(epochs, history['val_loss'], label='ошибка проверки')
    plt.title(f'ошибка: словарь={vocab_size}, окно={win_size}, шаг={win_hop}')
    plt.xlabel('эпоха')
    plt.ylabel('ошибка')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show(block=False)
    plt.close()


# запуск одного опыта
def run_experiment(vocab_size, win_size, win_hop, epochs=12, batch_size=128):
    # очистка памяти видеокарты между опытами
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    processor = DataProcessor(vocab_size, win_size, win_hop)
    x_train_bow, y_train, x_test_bow, y_test = processor.prepare(text_train, text_test)

    model = create_model(
        input_dim=x_train_bow.shape[1],
        class_count=len(CLASS_LIST)
    )

    history = train_model(
        model=model,
        x_train=x_train_bow,
        y_train=y_train,
        x_test=x_test_bow,
        y_test=y_test,
        epochs=epochs,
        batch_size=batch_size
    )

    plot_history(history, vocab_size, win_size, win_hop)

    result = {
        'VOCAB_SIZE': vocab_size,
        'WIN_SIZE': win_size,
        'WIN_HOP': win_hop,
        'Точность на обучении': history['accuracy'][-1],
        'Точность на проверке': history['val_accuracy'][-1],
        'Лучшая точность на проверке': max(history['val_accuracy']),
        'Ошибка на обучении': history['loss'][-1],
        'Ошибка на проверке': history['val_loss'][-1],
        'Лучшая ошибка на проверке': min(history['val_loss'])
    }

    return result, history


In [ ]:
# список параметров для 9 моделей
params = [
    (10000, 1000, 100),
    (10000, 1500, 250),
    (10000, 2000, 500),
    (15000, 1000, 100),
    (15000, 1500, 250),
    (15000, 2000, 500),
    (20000, 1000, 100),
    (20000, 1500, 250),
    (20000, 2000, 500),
]

results = []
histories = []

EPOCHS = 12
BATCH_SIZE = 128

start_time = time.time()

for number, (vocab_size, win_size, win_hop) in enumerate(params, start=1):
    print('\n' + '=' * 80)
    print(f'модель {number} из {len(params)}')
    print(f'словарь={vocab_size}, окно={win_size}, шаг={win_hop}')
    print('=' * 80)

    result, history = run_experiment(
        vocab_size=vocab_size,
        win_size=win_size,
        win_hop=win_hop,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE
    )

    results.append(result)
    histories.append({
        'label': f'{vocab_size}, {win_size}, {win_hop}',
        'history': history
    })

print('Время выполнения, сек.:', round(time.time() - start_time, 2))


In [ ]:
# итоговая таблица
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Лучшая точность на проверке', ascending=False).reset_index(drop=True)

display(results_df)

best_row = results_df.iloc[0]
print('Лучшая модель:')
print('VOCAB_SIZE:', best_row['VOCAB_SIZE'])
print('WIN_SIZE:', best_row['WIN_SIZE'])
print('WIN_HOP:', best_row['WIN_HOP'])
print('Лучшая точность на проверке:', round(best_row['Лучшая точность на проверке'], 4))
print('Лучшая ошибка на проверке:', round(best_row['Лучшая ошибка на проверке'], 4))
